# Mise en place d’une méthode pour la visualisation des représentations internes basées sur des réseaux convolutifs
CNAM certification pro IA - RCP 209 - Semestre 1 2024/2025 - Nicolas Guillard

Projet basé sur l'article "Zeiler and al. 2013". Ce carnet accompagne le rapport.

<a href="https://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC-SA 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1" alt=""></a>

## Modules

Chargement des modules prédéfinis nécessaires

In [ ]:
import os
import random
from typing import Tuple
from datetime import datetime

import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from torchinfo import summary
from PIL import Image, ImageDraw
import pandas as pd

Chargement des librairies définies pour ce projet

In [ ]:
from datasets import imagenet_mean, imagenet_std, \
DATASET_1, DATASET_3, CustomImageDataset, get_label_data_from_filename
from utils.convnet_wrapper_for_deconvolution import ConvnetWrapperForDeconvolution
from utils.utils_cnn import get_output_sizes, get_receptive_field_in_pixel_space
from utils.utils_images import display_image_tensor as display_image_tensor_, to_0_1, display_images_list_grid,   show_display_image_tensor_grid  
from utils.topk import TopK
from utils.deconvnet import Deconvnet
from imagenet_labels import imagenet1K_labels_to_names, imagenet1K_labels_to_codes

## Variables de contrôle

Ensemble de variables de contrôle d'exécution de ce carnet

In [ ]:
#DATASET = NONE # Choix du dataset
DATASET = DATASET_3 # Choix du dataset pour une utilisation locale # Imagenet-1K 50k images

SEED = 42 # graine aléatoire pour la reproductibilité

# Sélection des couches et du nombre de neurones à surveiller par couche
# Indexation en base 0 : sortie de la première couche opératoire du modèle
probed_layer_idx = [2, 5, 7, 9, 12] # couche cachée sélectionnée
probed_neuron_by_layer = [9, 16, 12, 10, 10] # nombre de neurones à surveiller pour chaque couche sélectionnée
K = 9 # Top K

# Taille d'un batch
batch_size = 32

# Choix du modèle
model_name = "alexnet"
TORCHVISION_MODELS_WEIGHTS = torchvision.models.AlexNet_Weights

# Paramètres pour la construction du deconvnet
flip_kernels = False
use_bias = False

# Pilotage de la convolution et déconvolution
i_stop = 0 # Stop après n batch si n = i_stop > 0
verbose_lvl = 0 # Niveau de verbosité pour la procédure de visualisation des représentations
receptive_field_color = "red" # Couleur du champ réceptif
save_session = True # Enregistrer les données liées aux activations et déconvolutions
#save_session_dir = "./deconv_sessions" # Répertoire de sauvegarde
save_session_dir = "/Users/me/Temp/CNAM/deconv_sessions" # Répertoire de sauvegarde
id_session = f"{model_name}_" + datetime.now().strftime("%Y%m%d_%H%M%S") # Identifiant de la session

## Paramétrages globaux

Pour la reproductibilité, paramétrage des processus aléatoires.

In [ ]:
random.seed(SEED);
torch.manual_seed(SEED); # Note : ne semble pas rendre torch.rand() invariant

## Récupération du jeu de données ImageNet-1K (jeu de validation du challenge ILSVRC 2012)

Récupération du jeu de données ImageNet-1K (validation) et des labels associés. Le jeu de données est divisé en 50 classes, chaque classe contenant 1000 images.

L'archive contenant ce jeu de données se trouve à `https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar`. Attention le téléchargement peut prendre un certain temps (> 30 min selon les bandes passantes du serveur et de votre connexion).

Les données de classification de vérité terrain sont disponibles dans le dictionnaire `imagebet1K_val_groundtruth_labels` définie dans le fichier `imagenet_labels.py` fourni, et obtenu par le traitement du fichier `ILSVRC2012_validation_ground_truth.txt` avec la table de correspondance via un script Matlab, tous deux disponibles dans https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz. La table de correspondance est disponible dans le fichier `ILSVRC2012_synsets.py` associé à ce carnet.

In [ ]:
use_wget = True # Utiliser wget pour télécharger le dataset, sinon curl, selon l'environnement
if DATASET == None:
    import re

    if not os.path.exists(save_session_dir):
        if use_wget:
            !wget https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar --no-check-certificate
        else:
            !curl -O https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar
        !mkdir ./ILSVRC2012_img_val
        !tar -xvf ILSVRC2012_img_val.tar -C ./ILSVRC2012_img_val

    DATASET = DATASET_3
    DATASET = {
        "path": "./ILSVRC2012_img_val",
        "mounted_path": None
    }


## Fonctions dédiées au carnet

Fonction d'afficage d'une image au format `torch.Tensor`.

In [ ]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(
        img_tensor, resize: Tuple[int, int] = None, resample: int = Image.Resampling.NEAREST, verbose=True
        ):
    if display:
        display_image_tensor_(img_tensor, resize=resize, resample=resample, verbose=verbose, fn_display=display)

Fonction superposant avec transparence une image de fond (l'image en entrée du modèle redimensionnée) et une image de premier plan (la représentation dans l'esoace de l'image de fond).

In [ ]:
def merge_input_and_deconv(background_image: torch.Tensor, output_deconv: torch.Tensor, receptive_field: Tuple[int, ...], alpha: float=0.75) -> Image.Image:
    assert alpha >= 0 and alpha <= 1, "Alpha must be between 0 and 1"

    ((top, left), (bottom, right)) = receptive_field
    background_image = T.functional.to_pil_image(background_image)
    output_deconv = T.functional.to_pil_image(output_deconv).crop((left, top, right, bottom))

    # Ajouter couche alpha pour superposer l'image de sortie sur l'image de fond
    background_image = background_image.convert("RGBA")
    output_resized = output_deconv.convert("RGBA")
    
    # Créer un masque pour la transparence (optionnel)
    mask = Image.new("L", output_resized.size, int(255 * alpha))

    # Coller l'image de sortie sur l'image de fond
    background_image.paste(output_resized, (left, top), mask)

    return background_image.convert("RGB")

## Plateforme d'exécution (CPU, GPU (cuda, mps), etc.)

Détection du GPU disponible.

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} est disponible")

## Les données

Création du `dataset`et du `dataloader`.

In [ ]:
# Transformations prédéfinies des images
transforms = TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1.transforms()

# méthode pour retrouber les données d'étiquetage à partir du nom de fichier
get_label_data = lambda f: get_label_data_from_filename(f, DATASET["name"])

dataset_path = DATASET["mounted_path"] \
    if (DATASET["mounted_path"] and os.path.exists(DATASET["mounted_path"]))\
          else DATASET["path"]

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True, # Le dataset sera utilisé par un DataLoader, donc certaines données ne seront pas retournées
    only_label_idx=False, # On a besoin de l'indice du fichier dans les données fournies par le DataLoader
    get_label_data=get_label_data,
    )

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Utilisation du dataset {DATASET['name']} situé dans {dataset_path} en contenant {len(dataset)} images")

## Le modèle ConvNet

Chargement du modèle ConvNet et création de l'enveloppe pour l'utiliser dans la procédure de visualisation des représentations internes.

In [ ]:
model_convnet = torch.hub.load('pytorch/vision', model_name, weights=TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1)
model_convnet.eval()
model_for_deconv = ConvnetWrapperForDeconvolution(model_convnet, model_convnet.features)

Résumé de la structure du modèle de type ConvNet

In [ ]:
#print("> Résumé de la structure du modèle de type ConvNet")
print(model_convnet)

Tableau des dimensionnalités des couches cachées du modèle

In [ ]:
#print("> Tableau des dimensionnalités des couches cachées du modèle")
batch_input = dataset[0][0].unsqueeze(dim=0)
print(batch_input.size())
model_for_deconv.set_return_switch_indices(False)
summary(model_convnet, input_size=batch_input.size(), mode="eval")

## Récupération des activations des neurones surveillés

Création des structures topK et choix aléatoire des coordonnées de $N$ neurones surveillés par couche cachée indiquée.

In [ ]:
# Récupération des tailles des sorties afin de pouvoir choisir des coordonnées aléatoires
# dans les sorties des couches
input_size=batch_input.size()
model_for_deconv.set_return_switch_indices(False)
output_sizes = get_output_sizes(model_for_deconv.convnet_features, input_size=input_size, last_2d=False)
print("Dimensions des différentes couches cachées")
for layer_idx, output_size in enumerate(output_sizes):
    print(f"\tcouche {layer_idx:2d} : {output_size}")

topk_activations_by_neuron = {}
coord_activations = {}
for layer_idx, n_neurons in zip(probed_layer_idx, probed_neuron_by_layer):
    coord_activations[layer_idx] = []
    i = 0
    # Randomly select N coordinates in the output of the layer : chanel, row, col
    while i < n_neurons:
        # Choisir une coordonnée aléatoire dans la sortie de la couche
        chn = random.randint(0, output_sizes[layer_idx][0]-1)
        row = random.randint(0, output_sizes[layer_idx][1]-1)
        col = random.randint(0, output_sizes[layer_idx][2]-1)
        # Vérifier que la coordonnée n'est pas déjà choisie (on ne veut pas de doublons)
        if (chn, row, col) not in coord_activations[layer_idx]:
            coord_activations[layer_idx].append((chn, row, col))
            i += 1
    topk_activations_by_neuron[layer_idx] = {coord: TopK(K) for coord in coord_activations[layer_idx]}
print(f"Coordonnées choisies : {coord_activations}")

Générations des activations

In [ ]:
BATCH_IMAGES_TRFM = 0
BATCH_LABEL_INDICES = 1
BATCH_FILE_INDICES = 3

if i_stop < 0:
    i_stop = min(len(dataloader) + i_stop, 1)

total = min(i_stop, len(dataloader)) if i_stop else len(dataloader)
all_activations = []
i_batch = 0
model_for_deconv.to(device)
with torch.no_grad():
    # Parcours du jeu de données par lot
    for batch in tqdm(dataloader, total=total, desc="Propagation", unit="batch"):
        # arrêt contrôlé par i_stop
        if i_stop and i_batch >= i_stop:
            break
        batch_input = batch[BATCH_IMAGES_TRFM].to(device)
        batch_label_idx = batch[BATCH_LABEL_INDICES]
        batch_input_file_idx = batch[BATCH_FILE_INDICES]

        # On fait passer le batch dans le modèle
        activations = model_for_deconv.get_activations(batch_input, coord_activations, verbose=False)
        # et on récupère les activations des couches sélectionnées
        all_activations.append((activations, batch_label_idx, batch_input_file_idx))

        i_batch += 1

Injection dans les structures topK

In [ ]:
for activations, batch_label_idx, batch_input_file_idx in all_activations:
    for idx_layer, activations_layer in activations.items():
            for coord, batch_activations_coord in zip(coord_activations[idx_layer], activations_layer):
                topk_activations_by_neuron[idx_layer][coord].append(
                     batch_activations_coord.tolist(),
                     list(zip(batch_input_file_idx.tolist(), batch_label_idx.tolist()))
                )

Affichage des topK et recherche du maximum des activations des neurones surveillés de chaque couche cachée sélectionnée.

In [ ]:
all_max_topk = {}
for idx_layer, coords in topk_activations_by_neuron.items():
    max_topk = 0
    max_topk_file_idx = None
    max_topk_label_idx = None
    max_coord = None
    for coord, topk in coords.items():
        print(f"> Layer {idx_layer} neuron {coord} :", topk)
        if topk[0][0] > max_topk:
            max_topk = topk[0][0]
            max_topk_file_idx, max_topk_label_idx = topk[0][1]
            max_coord = coord
    all_max_topk[idx_layer] = (max_topk, max_topk_file_idx, max_coord, max_topk_label_idx)

In [ ]:
for idx_layer, max_topk in all_max_topk.items():
    max_topk, max_topk_file_idx, max_coord, max_label_idx = max_topk
    print(f"Couche {idx_layer} - neurone {max_coord} : {max_topk} - Fichier : {max_topk_file_idx} - classe : {imagenet1K_labels_to_names[max_label_idx]}({max_label_idx})")

## Visualisation des représentations internes par déconvolution

### Création du DeconvNet

Création du DeconvNet qui va permettre de déconvoluer les activations des neurones surveillés pour les visualiser, et affichage de sa structure.

In [ ]:
deconvnet = Deconvnet(model_for_deconv, flip_kernels=flip_kernels, use_bias=use_bias)
print(deconvnet)

### Eléments Utiles

In [ ]:
# Opération inverse de la normalisation appliquée sur les images en entrée du ConvNet.
inv_mean = [-m/s for m, s in zip(imagenet_mean, imagenet_std)]
inv_std = [1/s for s in imagenet_std]
inv_normalize = T.Normalize(inv_mean, inv_std)
#inv_normalize = T.Normalize(
    #mean=[-0.485/0.229, -0.456/0.224, -0.406/0.255],
    #std=[1/0.229, 1/0.224, 1/0.255]
#)
#inv_normalize = UnNormalize(mean=imagenet_mean, std=imagenet_std) # Ne fonctionne pas !?

# Pour transformer les images PIL en tenseurs
pil_to_tensor = T.PILToTensor()

# Pour appliquer les transformations géométriques aux images du jeu de données pour retrouver la taille de l'image fournie en entrée du ConvNet
geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

### Procédure appliquée à l'ensemble des activations des neurones surveillés

In [ ]:
# Si sauvegarde des donnéées d'activation et de déconvolution
if save_session:
    os.makedirs(os.path.join(save_session_dir, id_session), exist_ok=True)
    data_activations = []
    _, image_trfm = dataset.get_image(0) # la taille de sortue de la déconvolution est la même que celle de l'image d'entrée image_trfm
    # taille du tenseur qui contiendra les différentes déconvolutions
    t_activations_size = (K, image_trfm.size(-3), image_trfm.size(-2), image_trfm.size(-1))

with torch.no_grad():
    deconvnet.wrapped_convnet.to(device)
    
    i_result = 0
    # Pour chaque couche surveillée
    for idx_layer, coords in tqdm(
        topk_activations_by_neuron.items(), 
        total=len(topk_activations_by_neuron),
        desc="Déconvolutions",
        unit="couche"
        ):
        # Pour chaque neurone surveillé dans la couche
        if save_session:
            t_activations = torch.zeros(t_activations_size)

        for i_neuron, (coord, topk) in enumerate(coords.items()):
            if verbose_lvl == 1:
                print(f"=== {model_name} - layer {idx_layer} - neuron {coord} - Top {K}===")
            idx_map, row, col = coord
            
            # Pour chaque activation du neurone surveillé dans la liste TopK
            for i_top, (top_value, (file_idx, label_idx)) in enumerate(topk):
                i_result += 1 # compteur d'activation sur l'ensemble des couches et neurones surveillés
                file_idx_b1 = file_idx + 1 # en base 1 pour l'affichage et l'enregistrement
                class_name = imagenet1K_labels_to_names[label_idx].lower() # nom de la classe de l'image associée à l'activation
                
                # Affichage d'information sur le neurone, l'activation et l'image asssociée
                info1 = f"(#{i_result:3d}) [{model_name} - L:{idx_layer} - n:{coord}] Top act({i_top+1})={top_value:.3f} avec img n° {file_idx_b1} ({label_idx}:{class_name})"
                if verbose_lvl == 1:
                    print(info1, end="|")

                # Récupération de l'image liée à l'activation
                image, image_trfm = dataset.get_image(file_idx)
                
                # Calcul du champ réceptif
                batch_input = image_trfm.unsqueeze(dim=0) # nécessaire pour get_output_sizes
                deconvnet.wrapped_convnet.to("cpu") # nécessaire pour get_output_sizes
                deconvnet.wrapped_convnet.set_return_switch_indices(False) # nécessaire pour get_output_sizes
                output_sizes = get_output_sizes(
                    deconvnet.wrapped_convnet.convnet_features, input_size=batch_input.size()
                ) # calcul des dimensions des couches cachées dans la partie "features" du ConvNet, nécessaire pour le calcul du champ réceptif
                pixel_space_size = batch_input.size(-2), batch_input.size(-1) # taille de l'image d'entrée
                receptive_field = get_receptive_field_in_pixel_space(
                    pos=(row, col),
                    idx_layer=idx_layer,
                    cnn_modules=model_for_deconv.convnet_features,
                    output_sizes=output_sizes,
                    pixel_space_size=pixel_space_size
                )

                # Dessin du champ réceptif sur l'image d'entrée
                ((top, left), (bottom, right)) = receptive_field
                # Affichage d'information sur le champ réceptif
                info2 = f"CR : {receptive_field} = {right-left+1} x {bottom-top+1}"
                if verbose_lvl == 1:
                    print(info2)
                image_resized = geo_transforms(image)
                batch_input = batch_input.squeeze(dim=0)
                image_resized_receptive_field = T.functional.to_pil_image(image_resized)
                img_draw = ImageDraw.Draw(image_resized_receptive_field)
                img_draw.rectangle([(left, top), (right, bottom)], outline=receptive_field_color)

                # Déconvolution
                deconvnet.wrapped_convnet.to(device)
                output_deconv = deconvnet.deconvolution(
                    batch_input.to(device),
                    idx_layer=idx_layer,
                    idx_map=idx_map,
                    pos=(row, col),
                    clean_feature_map=True,
                    return_pos=False,
                    verbose=verbose_lvl == 2
                    ).detach().cpu()
                
                # Transformation to put deconv output in the pixel space
                output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv))
                
                # Dession du champ réceptif sur l'image de sortie de la déconvolution
                output_deconv_receptive_field = T.functional.to_pil_image(output_deconv_in_pixel_space)
                img_draw = ImageDraw.Draw(output_deconv_receptive_field)
                img_draw.rectangle([(left, top), (right, bottom)], outline=receptive_field_color)

                # Superposition de l'image d'entrée et de la sortie de la déconvolution contenu dans le champ réceptif
                input_and_deconv = merge_input_and_deconv(
                    image_resized, output_deconv_in_pixel_space, receptive_field, alpha=0.95
                    )

                # Création et affichage de la combinaison d'images pour cette activation
                images = [
                    image_resized,
                    pil_to_tensor(image_resized_receptive_field),
                    pil_to_tensor(output_deconv_receptive_field),
                    image_resized[:, top:bottom+1, left:right+1],
                    output_deconv_in_pixel_space[:, top:bottom+1, left:right+1],
                    pil_to_tensor(input_and_deconv),
                    ]
                title = info1 + "|" + info2
                _, _ = display_images_list_grid(images, per_rows=6, figsize=(8, 2), title=title, no_plt_show=True);

                if save_session:
                    # Enregistrement des activations et des champs réceptifs
                    data_activations.append(
                        (i_result, idx_layer, i_neuron, coord, i_top+1, top_value, receptive_field, file_idx_b1, label_idx)
                        )
                    t_activations[i_top] = output_deconv

            show_display_image_tensor_grid() # car no_plt_show=True pour display_images_list_grid
            if save_session:
                 torch.save(t_activations, os.path.join(save_session_dir, id_session, f"deconvs_neuron_L{idx_layer:02d}_n{i_neuron:03d}.pt"))

if save_session:
    df = pd.DataFrame(
        data_activations,
        columns=["ID Act", "ID couche", "ID neurone", "Coord neurone", "ID TopK", "Activation", "Champ réceptif", "ID Fichier", "Classe"]
        )
    df.to_csv(os.path.join(save_session_dir, id_session, f"{id_session}_.csv"), index=False)
